# Chat Completion Audit Mask：三層去識別化

這本 Notebook 用兩個簡單案例觀察 CILLM Portal 的 Audit 去識別化流程：

1. 單純呼叫 Chat Completion。
2. 第一層：Portal 本地 Regex 預遮罩。
3. 第二層：Guardrails Container 內的 Presidio／spaCy NER（只說明效果，不在學員電腦安裝）。
4. 第三層：實際呼叫 `POST /v1/guardrails/mask`。

> 教材中的姓名、電話、Email、身分證字號均為虛構測試資料，請勿放入真實個資或正式 API Key。
> Chat Completion 經 Portal 時會自動留下 Audit；本 Notebook 不查詢 Oracle，只示範 Audit 寫入前使用的 Mask 效果。

In [ ]:
import json
import os
import re
import unicodedata
from pathlib import Path

import requests
from dotenv import load_dotenv
from openai import OpenAI


def find_lecture_root(start: Path) -> Path:
    candidates = [start, *start.parents, start / "Lecture04", start / "CILLM_Workshop" / "Lecture04"]
    for candidate in candidates:
        if candidate.name == "Lecture04" and (candidate / "03_audit_mask").is_dir():
            return candidate.resolve()
    raise RuntimeError("找不到 Lecture04 教材目錄，請從 CILLM_Workshop 或 Lecture04 啟動 Notebook。")


LECTURE_ROOT = find_lecture_root(Path.cwd().resolve())
NOTEBOOK_DIR = LECTURE_ROOT / "03_audit_mask"
load_dotenv(LECTURE_ROOT / ".env", override=False)
load_dotenv(NOTEBOOK_DIR / ".env", override=False)
os.chdir(NOTEBOOK_DIR)

API_KEY = os.getenv("CILLM_API_KEY", "").strip()
BASE_URL = (os.getenv("CILLM_BASE_URL") or "https://cillmtest.china-airlines.com/v1").strip().rstrip("/")
MODEL = (os.getenv("MODEL_NAME") or "openai/gpt-oss-120b").strip()
CHAT_MAX_TOKENS = int((os.getenv("CILLM_AGENT_MAX_TOKENS") or "2048").strip())
MASK_HTTP_TIMEOUT = int((os.getenv("GUARDRAIL_HTTP_TIMEOUT") or "180").strip())

if not API_KEY:
    raise RuntimeError("缺少 CILLM_API_KEY。請確認 Lecture04/.env 已正確設定。")

CILLM_HEADERS = {
    "X-User-ID": os.getenv("CILLM_USER_ID", "workshop-user"),
    "X-Platform": os.getenv("CILLM_PLATFORM", "cillm-workshop"),
    "X-Agent": os.getenv("CILLM_AGENT", "lecture04-audit-mask"),
}

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
    default_headers=CILLM_HEADERS,
    timeout=600,
)
MASK_URL = BASE_URL + "/guardrails/mask"

print(f"Lecture04 = {LECTURE_ROOT}")
print(f"Chat endpoint = {BASE_URL}/chat/completions")
print(f"Model = {MODEL}")
print(f"Mask endpoint = {MASK_URL}")

## Step 1：單純呼叫 Chat Completion

我們準備兩個案例：

- **案例 A：不需要去識別化**——一般航班數學問題。
- **案例 B：需要去識別化**——包含虛構姓名、手機、Email 與身分證字號。

這一步先觀察原始輸入與模型原始回答。請注意：經過 Portal 的 Chat Completion 已經會自動進入 Audit 流程。

In [ ]:
CASES = {
    "no_mask": {
        "label": "案例 A｜不需要去識別化",
        "prompt": "一架飛機有 312 個座位，載客率是 87%，請用一句話回答大約有多少位旅客。",
    },
    "needs_mask": {
        "label": "案例 B｜需要去識別化（全為虛構資料）",
        "prompt": (
            "請將以下虛構測試資料整理成一句話，欄位都要保留："
            "姓名是王小明，手機 0912-345-678，Email test.user@example.com，"
            "身分證字號 A123456789。"
        ),
    },
}


def chat_completion(prompt: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "請使用繁體中文，精簡回答，不要加入未提供的資料。"},
            {"role": "user", "content": prompt},
        ],
        max_tokens=CHAT_MAX_TOKENS,
        temperature=0,
        stream=False,
    )
    choice = response.choices[0]
    content = choice.message.content
    if not content:
        reasoning = getattr(choice.message, "reasoning", None)
        raise RuntimeError(
            "Chat Completion 沒有回傳 content。"
            f" finish_reason={choice.finish_reason}, reasoning_length={len(reasoning or '')}"
        )
    return content


for case in CASES.values():
    case["response"] = chat_completion(case["prompt"])
    print("=" * 70)
    print(case["label"])
    print("User：", case["prompt"])
    print("Assistant：", case["response"])

## Step 2：第一層——Regex 預遮罩

Portal 寫入 Audit 前會先做 NFKC 正規化，再依 `portal/config/audit_mask_rules.yml` 執行 Regex。
即使背景 Guardrails 暫時失敗，Oracle 至少仍保留 `[PRE_MASK]` 加上 Regex 遮罩後的安全版本。

下面只取姓名、手機、Email、身分證與 CILLM API Key 等少數規則作為課堂簡化版；正式 Portal 的 YAML 規則更多。

In [ ]:
REGEX_RULES = [
    (re.compile(r"cillm_sk_[A-Za-z0-9_\-]{3,}"), "[API_KEY]"),
    (re.compile(r"(?<![A-Za-z0-9])[A-Z][1289]\d{8}(?!\d)"), "[身分證]"),
    (re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"), "[EMAIL]"),
    (re.compile(r"(?<!\d)09\d{2}[- ]?\d{3}[- ]?\d{3}(?!\d)"), "[手機]"),
    (re.compile(r"(姓名(?:是|為|[:：]?)?)([一-龥]{2,3})(?=[，。；、,;.!?:\s]|$)"), r"\1[姓名]"),
]


def regex_mask(text: str) -> str:
    masked = unicodedata.normalize("NFKC", text)
    for pattern, replacement in REGEX_RULES:
        masked = pattern.sub(replacement, masked)
    return masked


for case in CASES.values():
    case["regex_prompt"] = regex_mask(case["prompt"])
    case["regex_response"] = regex_mask(case["response"])
    print("=" * 70)
    print(case["label"])
    print("Regex User：", case["regex_prompt"])
    print("Regex Assistant：", case["regex_response"])

## Step 3：第二層——Presidio／spaCy NER（效果展示，不在本機執行）

這一層執行在已部署的 **CILLM Guardrails Container**，不是執行在學員的華航內部電腦。
因此本章不提供 spaCy 安裝或下載程式碼，也不需要把 `en_core_web_sm` 放進學員環境。

NER 能依語意辨認 Regex 不容易完整描述的實體，例如英文人名：

| 輸入範例 | spaCy／Presidio 辨識後的概念效果 |
|---|---|
| `Passenger Alice Chen requested assistance.` | `Passenger [姓名] requested assistance.` |
| `Contact alice@example.com for details.` | `Contact [EMAIL] for details.` |
| `Call 0912345678 when ready.` | `Call [手機] when ready.` |

Guardrails Container 若無法載入 spaCy model，仍會降級使用 Presidio Pattern Recognizer 與 Regex，不會要求學員電腦連外下載。

## Step 4：第三層——實際呼叫 Guardrail Mask API

Portal 對外路徑為：

```http
POST /v1/guardrails/mask
Authorization: Bearer <CILLM_API_KEY>
Content-Type: application/json

{"text": "要去識別化的文字"}
```

回傳欄位是 `masked_text`。直接呼叫此 API 的 Key 必須具備 `guardrail.manage` scope。

In [ ]:
def guardrail_mask(text: str) -> str:
    try:
        response = requests.post(
            MASK_URL,
            headers={
                "Authorization": f"Bearer {API_KEY}",
                "Content-Type": "application/json",
                **CILLM_HEADERS,
            },
            json={"text": text},
            timeout=MASK_HTTP_TIMEOUT,
        )
    except requests.RequestException as exc:
        raise RuntimeError(f"無法連線 Guardrail Mask API：{MASK_URL}") from exc

    if response.status_code in (401, 403):
        raise RuntimeError(
            f"HTTP {response.status_code}：API Key 無效、過期，或缺少 guardrail.manage scope。"
        )
    if response.status_code == 404:
        raise RuntimeError(f"HTTP 404：Portal 沒有此 Mask 路徑：{MASK_URL}")
    if response.status_code == 502:
        raise RuntimeError(f"HTTP 502：Guardrail Mask 執行失敗。response={response.text[:500]}")
    if not response.ok:
        raise RuntimeError(f"Mask API HTTP {response.status_code}：{response.text[:500]}")

    try:
        payload = response.json()
    except requests.JSONDecodeError as exc:
        raise RuntimeError(f"Mask API 未回傳 JSON：{response.text[:500]}") from exc
    masked_text = payload.get("masked_text")
    if not isinstance(masked_text, str):
        raise RuntimeError(f"Mask API 回應缺少 masked_text：{payload}")
    return masked_text

In [ ]:
for case in CASES.values():
    case["api_prompt"] = guardrail_mask(case["prompt"])
    case["api_response"] = guardrail_mask(case["response"])
    print("=" * 70)
    print(case["label"])
    print("Mask API User：", case["api_prompt"])
    print("Mask API Assistant：", case["api_response"])

## 最後比較

下面把每個案例的原文、Regex 與 Mask API 結果放在一起。正常內容應大致維持不變；含個資內容應出現 `[姓名]`、`[手機]`、`[EMAIL]`、`[身分證]` 等標記。

In [ ]:
for case in CASES.values():
    print("#" * 80)
    print(case["label"])
    print("[User 原文]      ", case["prompt"])
    print("[User Regex]     ", case["regex_prompt"])
    print("[User Mask API]  ", case["api_prompt"])
    print("-" * 80)
    print("[AI 原文]        ", case["response"])
    print("[AI Regex]       ", case["regex_response"])
    print("[AI Mask API]    ", case["api_response"])

## 小結

Portal 的 Audit 去識別化不是只靠單一模型：

```text
Chat Completion request / response
        ↓
Portal slim + Regex simple_mask
        ↓  先 INSERT Oracle：[PRE_MASK] 安全版本
Guardrail Mask API（Presidio／spaCy + Pattern + Regex）
        ↓  成功後背景 UPDATE
Oracle 最終去識別版本
```

Guardrail 暫時失敗時，Audit 仍保留 Regex 預遮罩版本；因此稽核紀錄不會因背景服務故障而完全失去最低限度保護。